#🧠**Desafío Individual: Sistema de Gestión de Obra Inteligente**

###❓**Contexto del Problema**
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* **Evaluación de Riesgos en Obra (Lógica Deductiva):** Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.

* **Planificación de Maquinaria Pesada (Satisfacción de Restricciones):** Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

###🧩**Misión A: Diagnóstico de Seguridad con experta**
Debes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un cerebro Lógico para evitar accidentes.

####☝**Reglas a Implementar:**
- **Crítico (Paro de Obra):**  la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.

- **Moderado (Precaución):** la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o hay humedad extrema en zonas de excavación.

- **Bajo (Operación Normal):** los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.
---
Consigna Técnica: Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

###🧩**Mision A**

In [ ]:
# Parche de compatibilidad e importación de experta.
import collections.abc; collections.Mapping = collections.abc.Mapping
from experta import *

In [ ]:
# Defino la clase para el hecho.
class EstadoObra(Fact):
  """Datos capturados por sensores en el sitio"""
  pass

In [ ]:
# Defino mi motor (base de conocimientos + motor de inferencia).
class MotorSeguridad(KnowledgeEngine):

  # 1. Crítico (Paro de Obra): viento > 60 o grietas
  @Rule(OR(
    EstadoObra(viento=P(lambda v: v > 60)),
    EstadoObra(grieta=True)), salience=100)
  def riesgo_critico(self):
    print("ALERTA - Nivel de Riesgo: Crítico (Paro de Obra)")
    self.halt() # Por ser una condición terminar (no importa seguir evaluando).

  # 2. Moderado (Precaución): viento entre 40 y 60 o humedad extrema.
  @Rule(OR(
    EstadoObra(viento=P(lambda v: 40<v<60)),
    EstadoObra(humedad=True)
  ))
  def riesgo_moderado(self):
    print("PRECAUCIÓN - Nivel de Riesgo: Moderado")

  # 3. Bajo (Operación Normal): viento < 40 y sin alertas
  @Rule(AND(
    EstadoObra(viento=P(lambda v: v < 40)),
    EstadoObra(alertas=False)
  ))
  def riesgo_bajo(self):
    print("NORMAL - Nivel de Riesgo: Bajo (Operación Normal)")


# Instanciar.
motor = MotorSeguridad()

# Reiniciar.
motor.reset()

# Declarar.
motor.declare(EstadoObra(viento=65, grieta=True))

# Ejecutar.
motor.run()

ALERTA - Nivel de Riesgo: Crítico (Paro de Obra)


###🧩**Misión B: Ubicación de Equipos con python-constraint**

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

####❌**Restricciones (Reglas de Oro):**
- **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).

- **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.

- **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.

- **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.


Consigna Técnica: Define las variables (Máquinas) y el dominio (Zonas de la obra).

---

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

###🧩**Mision B**

In [ ]:
# Instalo la librería python-constrint.
!pip install python-constraint
from constraint import *

In [ ]:
# Instanciar.
problema = Problem()

In [ ]:
# Variables y Dominios.
problema.addVariable("grua_torre", ["zona_estable"])
problema.addVariable("excavadora", ["zona_estable", "zona_general"])
problema.addVariable("hormiguera", ["zona_estrecha", "zona_general"])

In [ ]:
# Restricciones,
problema.addConstraint(AllDifferentConstraint(), ['grua_torre', 'excavadora', 'hormiguera'])

In [ ]:
# Resolución.
solucion = problema.getSolution()
print(solucion)

{'grua_torre': 'zona_estable', 'excavadora': 'zona_general', 'hormiguera': 'zona_estrecha'}


##✉ **Parámetros de Entrega y Evaluación**
El entregable debe cumplir con lo siguiente:

- **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.

- **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

---

Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.

**¿Por qué un modelo de grafos/árboles (Sistemas Expertos) es superior a una lista de if/else?**


1. Escalabilidad y Mantenimiento: en los sistemas basados en reglas, cada regla es independiente; podemos agregar, modificar o eliminar restricciones sin tocar el resto del código.


2. Eficiencia Computacional (Algoritmo Rete): los motores de inferencia modernos construyen una red (un grafo dirigido) en memoria. Cuando un dato cambia (ej. sube el viento), el motor no reevalúa todo desde cero, sino que el dato fluye por el grafo activando únicamente los nodos correspondientes.


3. Manejo de Conflictos: Si dos condiciones if son verdaderas, el programa ejecutará solo la primera que encuentre y descartará el resto. Los sistemas expertos permiten asignar "prioridades" (Salience) y manejar múltiples reglas que se cumplen en simultáneo de forma inteligente.